In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Setup & Installations

In [1]:
!pip install -q git+https://github.com/huggingface/transformers.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 18.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 99.6 MB/s eta 0:00:00:00:01


In [2]:
!pip install -q accelerate bitsandbytes sentencepiece langchain_core langchain_huggingface pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 30.8 MB/s eta 0:00:00


In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secret_label = "HF_FACE"
secret_value = UserSecretsClient().get_secret(secret_label)
 
login(secret_value)

# Imports 

In [23]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Optional, List
import json

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

## Schema Configuration

In [5]:
class TestScores(BaseModel):
    """Schema for standardized test scores"""
    ielts: Optional[float] = Field(None, description="IELTS band score (0-9)")
    toefl: Optional[int] = Field(None, description="TOEFL score (0-120)")
    duolingo: Optional[int] = Field(None, description="Duolingo score (10-160)")
    gre_verbal: Optional[int] = Field(None, description="GRE Verbal Reasoning score")
    gre_quant: Optional[int] = Field(None, description="GRE Quantitative Reasoning score")
    gre_awa: Optional[float] = Field(None, description="GRE Analytical Writing score")


class UserProfile(BaseModel):
    """The final cleaned profile of the scholarship applicant"""
    university: Optional[str] = Field(None, description="University name")
    degree_and_specialty: Optional[str] = Field(None, description="Degree and major, e.g., BSc Computer Science")
    gpa: Optional[float] = Field(None, description="Grade Point Average")
    gpa_scale: Optional[float] = Field(4.0, description="GPA scale, e.g., 4.0 or 5.0")
    test_scores: TestScores = Field(default_factory=TestScores)
    project_titles: List[str] = Field(default_factory=list, description="List of project titles")
    published_research_titles: List[str] = Field(default_factory=list, description="List of published research paper titles")
    competition_wins: List[str] = Field(default_factory=list, description="List of competitions won")
    volunteering_activities: List[str] = Field(default_factory=list, description="List of volunteering activities")

## Model Loading

In [6]:
MODEL_NAME="google/gemma-2-9b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config, # to prevent OOM
    device_map="auto"               
)

print("-- Model Loaded --")

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

-- Model Loaded --


In [24]:
pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    repetition_penalty=1.2,
    return_full_text=False
)

llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipeline))

# Agent1 Data Extractor

## Chain Logic

In [45]:
parser = PydanticOutputParser(pydantic_object=UserProfile)

# FIX: Removed ("system", ...) and merged everything into ("human", ...) for Gemma 2
SYSTEM_PROMPT = ChatPromptTemplate.from_messages([
    ("human", """You are an expert Admissions Profiler AI. Your job is to extract specific information from a user's resume or profile text and format it as JSON.

Extraction Rules:
1. Only extract information explicitly stated in the text. Do NOT guess or hallucinate.
2. If a field is not mentioned, return null for strings/numbers, and an empty array [] for lists.
3. For GPA, also try to identify the scale (e.g., 4.0 or 5.0). If not stated, assume 4.0.
4. Extract the EXACT titles of projects, research papers, competitions, and volunteering. Do not summarize them.

{format_instructions}

Rules: 
Respond ONLY with valid JSON.
Do not include explanations or markdown.

Here is the applicant's data:
{applicant_text}""")
])

## Clean Json 

In [46]:
import re 

In [47]:
def clean_json_output(output_string: str) -> str:
    output_string = output_string.replace("```json", "").replace("```", "")
    start = output_string.find('{')
    end = output_string.rfind('}') + 1
    if start != -1 and end > start:
        return output_string[start:end]
    return output_string

In [48]:
def extract_json(text: str) -> str:
    """Finds the first valid JSON object ({...}) in a string."""
    # Remove markdown code blocks if the LLM added them
    text = text.replace("```json", "").replace("```", "")
    
    # Find the first { and the last }
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        return match.group(0)
    else:
        raise ValueError("No JSON object found in the LLM output.")

In [49]:
chain1 = (
    SYSTEM_PROMPT.partial(
        format_instructions=parser.get_format_instructions()
    )
    | llm
    | StrOutputParser() 
    | RunnableLambda(clean_json_output))

print("-- chain created successfully! --")

-- chain created successfully! --


## Extract text from PDF

In [50]:
def extract_text_from_pdf(file_bytes) -> str:
    """Reads PDF bytes and returns cleaned text."""
    try:
        pdf_file = io.BytesIO(file_bytes)
        reader = PdfReader(pdf_file)
        raw_text = ""
        for page in reader.pages:
            raw_text += page.extract_text() + "\n"

        clean_text = re.sub(r'\s+', ' ', raw_text).strip()
        return clean_text
    except Exception as e:
        print(f"Error reading PDF: {e}")
        return ""

In [51]:
def process_agent_input(input_type, text_data, file_bytes):
    """Core function to process either text or PDF and run the chain"""
    if input_type == 'Upload PDF':
        if not file_bytes:
            return "Please upload a PDF file first."
        text_data = extract_text_from_pdf(file_bytes)
        if not text_data:
            return "Could not extract text from PDF. Is it a scanned image?"
    
    if not text_data.strip():
        return "No data provided. Please type text or upload a PDF."

    # Truncate if text is too long for the model context
    if len(text_data) > 4000:
        text_data = text_data[:4000]

## Test text 

In [52]:
sample_text = """
Hi, my name is Abdallah. 
I graduated from the University of Toronto with a BSc in Computer Science.
My GPA is 3.8 out of 4.0. 
I took the TOEFL and scored 112. 
Also did the GRE and got Verbal 162, Quant 170, AWA 4.5.
For projects, I built a "Real-time Sign Language Translator" and a "Decentralized Voting System using Blockchain". 
I have two published papers: "Optimizing LSTM for Real-Time Video Processing" and "Blockchain Consensus Mechanisms in IoT Networks". 
I won 1st place in the Google Hackathon 2023 and got a Bronze medal in the ICPC Regional Finals. 
In my free time, I volunteer at Code.org teaching kids to code, and I also help out at the local animal shelter.

"""

print("\nRunning Agent 1")
try:
    raw_output = chain1.invoke({"applicant_text": sample_text})
    if hasattr(raw_output, "content"):
        profile = raw_output.content
    
    cleaned = extract_json(raw_output)

    data = json.loads(cleaned)

    profile = UserProfile(**data)

    
    print("\n-- EXTRACTION SUCCESSFUL! --")
    print("-" * 30)
    print(f"University:          {profile.university}")
    print(f"Degree:              {profile.degree_and_specialty}")
    print(f"GPA:                 {profile.gpa}/{profile.gpa_scale}")
    print(f"TOEFL:               {profile.test_scores.toefl}")
    print(f"GRE (V/Q/AWA):       {profile.test_scores.gre_verbal} / {profile.test_scores.gre_quant} / {profile.test_scores.gre_awa}")
    print(f"Projects:            {profile.project_titles}")
    print(f"Research Titles:     {profile.published_research_titles}")
    print(f"Competitions:        {profile.competition_wins}")
    print(f"Volunteering:        {profile.volunteering_activities}")
    
except Exception as e:
    print(f"\n-- EXTRACTION FAILED: {e} --")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Running Agent 1
🔍 DEBUG: RAW LLM OUTPUT (Look closely here!)


-- EXTRACTION SUCCESSFUL! --
------------------------------
University:          University of Toronto
Degree:              BSc Computer Science
GPA:                 3.8/4.0
TOEFL:               112
GRE (V/Q/AWA):       162 / 170 / 4.5
Projects:            ['Real-time Sign Language Translator', 'Decentralized Voting System using Blockchain']
Research Titles:     ['Optimizing LSTM for Real-Time Video Processing', 'Blockchain Consensus Mechanisms in IoT Networks']
Competitions:        ['Google Hackathon 2023 - 1st Place', 'ICPC Regional Finals - Bronze Medal']
Volunteering:        ['Code.org - Teaching kids to code', 'Local Animal Shelter']


## Test pdf

In [53]:
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
import io
import re
import time

In [54]:
print("Please upload a CV/Resume PDF:")
uploader = widgets.FileUpload(
    accept='.pdf',  
    multiple=False  # Only one file at a time
)
display(uploader)

Please upload a CV/Resume PDF:


FileUpload(value=(), accept='.pdf', description='Upload')

In [56]:
import time
import json

# Wait for upload
while not uploader.value:
    print("Waiting for file upload...")
    time.sleep(2)

print("File uploaded! Extracting text from PDF...")

# Get bytes
uploaded_file = uploader.value[0]
file_bytes = uploaded_file['content']

pdf_text = extract_text_from_pdf(file_bytes)

if pdf_text:
    # print(f"\nPreview (first 300 chars):\n{pdf_text[:300]}...\n")
    
    print("Running Agent 1: Asking LLM to extract data...")
    
    raw_llm_output = chain1.invoke({"applicant_text": pdf_text})

    print("Attempting to parse into Pydantic object...")
    try:
        profile = UserProfile.model_validate_json(raw_llm_output)
        
        print("\nEXTRACTION SUCCESSFUL FROM PDF!")
        print("-" * 30)
        print(f"University:          {profile.university}")
        print(f"Degree:              {profile.degree_and_specialty}")
        print(f"GPA:                 {profile.gpa}/{profile.gpa_scale}")
        print(f"TOEFL:               {profile.test_scores.toefl}")
        print(f"Projects:            {profile.project_titles}")
        
    except Exception as e:
        print(f"\nPARSING FAILED: {e}")
        print("The LLM did not return valid JSON that matches our schema.")
else:
    print("Could not extract text from the PDF.")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


File uploaded! Extracting text from PDF...
Running Agent 1: Asking LLM to extract data...
Attempting to parse into Pydantic object...

EXTRACTION SUCCESSFUL FROM PDF!
------------------------------
University:          Faculty of Computers and Informatics, Suez Canal University
Degree:              Computer Science
GPA:                 3.2/4.0
TOEFL:               None
Projects:            ['Ai Portfolio', 'Chatbot', 'Face Recognition', 'Sentiment Analysis']
